## RAG 第 3 天

### 1topcloud LLM 的专家问答系统

基于 LangChain 1.0 实现的 RAG 管道。

使用我们上次创建的 VectorStore（配合 HuggingFace `all-MiniLM-L6-v2`）

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### 连接到 Chroma；使用 Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### 设置 2 个关键的 LangChain 对象：retriever（检索器）和 llm（大语言模型）

#### 关于“temperature”（温度）的旁注：
- 控制输出的多样性
- temperature 为 0 意味着输出应该是可预测的
- 较高的 temperature 会使答案更加多样化

有些人将 temperature 描述为类似“创造力”，但这并不完全正确
- 它实际上控制着推理过程中哪些 token（词元）被选中
- temperature=0 意味着：始终选择概率最高的 token
- temperature=1 通常意味着：有 10% 概率的 token 应该有 10% 的时间被选中

注意：temperature 为 0 并不意味着输出总是可复现的。你还需要设置随机种子。我们将在第 6-8 周做这件事。（即使那样，也不总是可复现的。）

注意 2：如果你想要创造力，请使用 System Prompt（系统提示词）！

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### 这些 LangChain 对象实现了 `invoke()` 方法

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='f021e980-a7af-4232-a98c-3862e1235b61', metadata={'doc_type': 'employees', 'source': '/Users/macbook/projects/Learning_LLM/第五周/knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing 1topcloud and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving 1topcloud's corporate social responsibility image.  \n\nAvery Lanc

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_2642d879e2', 'id': 'chatcmpl-DFt4ivHuftrlcpTwtlLOoYTXQ3e4D', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbbe6-e8dc-7180-a5f5-1a33a40c5931-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 24, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 是时候把它们整合起来了！

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about 1topcloudllm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("Who is averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of 1topcloud, a leading Insurance Tech company. She co-founded the company in 2015 and has been instrumental in its growth and innovation within the insurance industry. Avery is known for her leadership, risk management expertise, and strategic vision that have helped position 1topcloud as a prominent player in the market.'

## 接下来会发生什么？ 😂

In [11]:
gr.ChatInterface(answer_question).launch()

/opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/chat_interface.py:338: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 承认吧 —— 你原以为 RAG 会比这更复杂！！